## Clustering with PySpark

In [1]:
!curl https://raw.githubusercontent.com/apache/spark/master/data/mllib/sample_kmeans_data.txt >> sample_kmeans_data.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   120  100   120    0     0    179      0 --:--:-- --:--:-- --:--:--   178


In [2]:
# !curl https://archive.ics.ucl.edu/ml/machine-learning-databases/00236/seeds_dataset.txt >> seeds_dataset.txt

### K-Means 

In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('sample_cluster').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/23 13:49:20 WARN Utils: Your hostname, aditya-HP-Laptop-15s-eq1xxx, resolves to a loopback address: 127.0.1.1; using 10.103.210.123 instead (on interface wlo1)
26/06/23 13:49:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/23 13:49:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

In [5]:
df = spark.read.format('libsvm').load('sample_kmeans_data.txt')

26/06/23 13:49:26 WARN LibSVMFileFormat: 'numFeatures' option not specified, determining the number of features by going though the input. If you know the number in advance, please specify it via 'numFeatures' option to avoid the extra scan.


In [6]:
df.show()

+-----+--------------------+
|label|            features|
+-----+--------------------+
|  0.0|           (3,[],[])|
|  1.0|(3,[0,1,2],[0.1,0...|
|  2.0|(3,[0,1,2],[0.2,0...|
|  3.0|(3,[0,1,2],[9.0,9...|
|  4.0|(3,[0,1,2],[9.1,9...|
|  5.0|(3,[0,1,2],[9.2,9...|
|  0.0|           (3,[],[])|
|  1.0|(3,[0,1,2],[0.1,0...|
|  2.0|(3,[0,1,2],[0.2,0...|
|  3.0|(3,[0,1,2],[9.0,9...|
|  4.0|(3,[0,1,2],[9.1,9...|
|  5.0|(3,[0,1,2],[9.2,9...|
|  0.0|           (3,[],[])|
|  1.0|(3,[0,1,2],[0.1,0...|
|  2.0|(3,[0,1,2],[0.2,0...|
|  3.0|(3,[0,1,2],[9.0,9...|
|  4.0|(3,[0,1,2],[9.1,9...|
|  5.0|(3,[0,1,2],[9.2,9...|
+-----+--------------------+



In [7]:
kmeans = KMeans().setK(2).setSeed(42)
model = kmeans.fit(df)

In [8]:
pred = model.transform(df)

In [9]:
evals = ClusteringEvaluator()

In [10]:
silhouette = evals.evaluate(pred)

print(f"Silhouette with squared euclidean distance: {silhouette}")

Silhouette with squared euclidean distance: 0.9998147729031402


In [11]:
centers = model.clusterCenters()
print("Cluster Centers:")
print("=================")

for center in centers:
    print(center)

Cluster Centers:
[0.1 0.1 0.1]
[9.1 9.1 9.1]


#### New example: Seed clustering data from UCI

In [12]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('seed').getOrCreate()

26/06/23 13:49:34 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [13]:
from pyspark.ml.clustering import KMeans

In [14]:
!head seeds_dataset.csv

15.26	14.84	0.871	5.763	3.312	2.221	5.22	1
14.88	14.57	0.8811	5.554	3.333	1.018	4.956	1
14.29	14.09	0.905	5.291	3.337	2.699	4.825	1
13.84	13.94	0.8955	5.324	3.379	2.259	4.805	1
16.14	14.99	0.9034	5.658	3.562	1.355	5.175	1
14.38	14.21	0.8951	5.386	3.312	2.462	4.956	1
14.69	14.49	0.8799	5.563	3.259	3.586	5.219	1
14.11	14.1	0.8911	5.42	3.302	2.7		5		1
16.63	15.46	0.8747	6.053	3.465	2.04	5.877	1
16.44	15.25	0.888	5.884	3.505	1.969	5.533	1


In [15]:
df = spark.read.option('delimiter', '\t').csv('seeds_dataset.csv', header=None, inferSchema=True).drop('_c7')

In [16]:
df.show()

+-----+-----+------+-----+-----+-----+-----+
|  _c0|  _c1|   _c2|  _c3|  _c4|  _c5|  _c6|
+-----+-----+------+-----+-----+-----+-----+
|15.26|14.84| 0.871|5.763|3.312|2.221| 5.22|
|14.88|14.57|0.8811|5.554|3.333|1.018|4.956|
|14.29|14.09| 0.905|5.291|3.337|2.699|4.825|
|13.84|13.94|0.8955|5.324|3.379|2.259|4.805|
|16.14|14.99|0.9034|5.658|3.562|1.355|5.175|
|14.38|14.21|0.8951|5.386|3.312|2.462|4.956|
|14.69|14.49|0.8799|5.563|3.259|3.586|5.219|
|14.11| 14.1|0.8911| 5.42|3.302|  2.7| NULL|
|16.63|15.46|0.8747|6.053|3.465| 2.04|5.877|
|16.44|15.25| 0.888|5.884|3.505|1.969|5.533|
|15.26|14.85|0.8696|5.714|3.242|4.543|5.314|
|14.03|14.16|0.8796|5.438|3.201|1.717|5.001|
|13.89|14.02| 0.888|5.439|3.199|3.986|4.738|
|13.78|14.06|0.8759|5.479|3.156|3.136|4.872|
|13.74|14.05|0.8744|5.482|3.114|2.932|4.825|
|14.59|14.28|0.8993|5.351|3.333|4.185|4.781|
|13.99|13.83|0.9183|5.119|3.383|5.234|4.781|
|15.69|14.75|0.9058|5.527|3.514|1.599|5.046|
| 14.7|14.21|0.9153|5.205|3.466|1.767|4.649|
|12.72|13.

In [17]:
df.printSchema()

root
 |-- _c0: double (nullable = true)
 |-- _c1: double (nullable = true)
 |-- _c2: double (nullable = true)
 |-- _c3: double (nullable = true)
 |-- _c4: double (nullable = true)
 |-- _c5: double (nullable = true)
 |-- _c6: double (nullable = true)



In [18]:
columns = ['area', 'perimeter', 'compactness', 'length_of_kernel', 'width_of_kernel', 'asymmetry_coefficient', 'length_of_groove']

mapping = dict.fromkeys(['_c' + str(i) for i in range(7)]) # initialized a dictionary with keys as old column names and values as None

for index, key in enumerate(mapping.keys()):
    mapping[key] = columns[index]

In [19]:
mapping

{'_c0': 'area',
 '_c1': 'perimeter',
 '_c2': 'compactness',
 '_c3': 'length_of_kernel',
 '_c4': 'width_of_kernel',
 '_c5': 'asymmetry_coefficient',
 '_c6': 'length_of_groove'}

In [20]:
df = df.withColumnsRenamed(mapping)

In [21]:
df.show()

+-----+---------+-----------+----------------+---------------+---------------------+----------------+
| area|perimeter|compactness|length_of_kernel|width_of_kernel|asymmetry_coefficient|length_of_groove|
+-----+---------+-----------+----------------+---------------+---------------------+----------------+
|15.26|    14.84|      0.871|           5.763|          3.312|                2.221|            5.22|
|14.88|    14.57|     0.8811|           5.554|          3.333|                1.018|           4.956|
|14.29|    14.09|      0.905|           5.291|          3.337|                2.699|           4.825|
|13.84|    13.94|     0.8955|           5.324|          3.379|                2.259|           4.805|
|16.14|    14.99|     0.9034|           5.658|          3.562|                1.355|           5.175|
|14.38|    14.21|     0.8951|           5.386|          3.312|                2.462|           4.956|
|14.69|    14.49|     0.8799|           5.563|          3.259|                3.58

In [22]:
df.describe().show()

26/06/23 13:49:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+--------------------+------------------+------------------+---------------------+------------------+
|summary|              area|         perimeter|         compactness|  length_of_kernel|   width_of_kernel|asymmetry_coefficient|  length_of_groove|
+-------+------------------+------------------+--------------------+------------------+------------------+---------------------+------------------+
|  count|               210|               210|                 207|               210|               209|                  210|               206|
|   mean|14.847523809523816|14.559285714285718|  0.8712797101449273| 5.563918095238097|  3.28144019138756|   3.6935295238095227| 5.407529126213591|
| stddev|2.9096994306873647|1.3059587265640225|0.023305838551641696|0.7195936127550641|0.4199071755390635|   1.4951121686440525|0.5323300430722294|
|    min|             10.59|             12.41|              0.8081|            0.8189|              2.63|      

#### Format Data

In [23]:
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import VectorAssembler

In [24]:
df.columns

['area',
 'perimeter',
 'compactness',
 'length_of_kernel',
 'width_of_kernel',
 'asymmetry_coefficient',
 'length_of_groove']

In [25]:
assembler = VectorAssembler(inputCols=df.columns, outputCol='features', handleInvalid='skip')

In [26]:
df_final = assembler.transform(df)

In [27]:
df_final.show()

+-----+---------+-----------+----------------+---------------+---------------------+----------------+--------------------+
| area|perimeter|compactness|length_of_kernel|width_of_kernel|asymmetry_coefficient|length_of_groove|            features|
+-----+---------+-----------+----------------+---------------+---------------------+----------------+--------------------+
|15.26|    14.84|      0.871|           5.763|          3.312|                2.221|            5.22|[15.26,14.84,0.87...|
|14.88|    14.57|     0.8811|           5.554|          3.333|                1.018|           4.956|[14.88,14.57,0.88...|
|14.29|    14.09|      0.905|           5.291|          3.337|                2.699|           4.825|[14.29,14.09,0.90...|
|13.84|    13.94|     0.8955|           5.324|          3.379|                2.259|           4.805|[13.84,13.94,0.89...|
|16.14|    14.99|     0.9034|           5.658|          3.562|                1.355|           5.175|[16.14,14.99,0.90...|
|14.38|    14.21

#### Scaling

In [28]:
from pyspark.ml.feature import StandardScaler

In [29]:
scaler = StandardScaler(inputCol='features', outputCol='scaledFeatures', withStd=True, withMean=False)

In [30]:
scaledModel = scaler.fit(df_final)

In [31]:
df_final = scaledModel.transform(df_final)

In [32]:
df_final.show()

+-----+---------+-----------+----------------+---------------+---------------------+----------------+--------------------+--------------------+
| area|perimeter|compactness|length_of_kernel|width_of_kernel|asymmetry_coefficient|length_of_groove|            features|      scaledFeatures|
+-----+---------+-----------+----------------+---------------+---------------------+----------------+--------------------+--------------------+
|15.26|    14.84|      0.871|           5.763|          3.312|                2.221|            5.22|[15.26,14.84,0.87...|[5.22628833884476...|
|14.88|    14.57|     0.8811|           5.554|          3.333|                1.018|           4.956|[14.88,14.57,0.88...|[5.09614485465335...|
|14.29|    14.09|      0.905|           5.291|          3.337|                2.699|           4.825|[14.29,14.09,0.90...|[4.89407997130352...|
|13.84|    13.94|     0.8955|           5.324|          3.379|                2.259|           4.805|[13.84,13.94,0.89...|[4.73996268739

#### Train and Evaluate

In [44]:
kmeans = KMeans(featuresCol='scaledFeatures', k=3)
model = kmeans.fit(df_final)

In [45]:
pred = model.transform(df_final)

In [46]:
from pyspark.ml.evaluation import ClusteringEvaluator

In [47]:
evals = ClusteringEvaluator()

In [48]:
silhouette = evals.evaluate(pred)

print(f"Silhouette with squared euclidean distance: {silhouette}")

Silhouette with squared euclidean distance: 0.603847180910434


In [50]:
centers = model.clusterCenters()

In [52]:
print("Cluster Centers:")
print("==================")
for center in centers:
    print(center)

Cluster Centers:
[ 4.84806185 10.84623024 37.70748986 12.33589015  8.52128482  1.81592511
 10.29444274]
[ 6.29473699 12.33425241 37.86587742 13.8957      9.72990669  2.39707377
 12.23479873]
[ 4.0567094  10.12026466 36.26673707 11.81839958  7.50008382  3.28922383
 10.41121669]
